# Scoring Pipeline — Geography of Rights

Loads the five dimension spreadsheets + predictor variables, normalizes all sub-indicators
to 0-100 (min-max), computes dimension scores (mean of normalized sub-indicators), and
produces two composite indices:

1. **Equal-weight composite** — each of the 5 dimensions contributes 20%.
2. **PCA-weighted composite** — weights derived from the first principal component of
   the 5 dimension scores (for empirical comparison).

Outputs `data/processed/master_scored.csv` containing raw + normalized sub-indicators,
dimension scores, both composites, and merged predictor variables for downstream
regression.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

ROOT = Path('.').resolve()
OUT_DIR = ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)

Project root: /Users/latahviawilliams/DSS_rights


## 1. Configuration

Each dimension maps to a source sheet and a list of sub-indicator column prefixes
(e.g. `"1.1"`). Prefixes are used because actual column names contain newlines and
score ranges.

In [2]:
DIMENSIONS = {
    'Reproductive Rights': {
        'file': 'role1_reproductive_rights.xlsx',
        'sheet': 'Reproductive Rights Data',
        'prefixes': ['1.1', '1.2', '1.3', '1.4', '1.5', '1.6'],
    },
    'LGBTQ+ Protections': {
        'file': 'role2_lgbtq_voting (1).xlsx',
        'sheet': 'LGBTQ+ Protections Data',
        'prefixes': ['2.1', '2.2', '2.3', '2.4', '2.5'],
    },
    'Voting Access': {
        'file': 'role2_lgbtq_voting (1).xlsx',
        'sheet': 'Voting Access Data',
        'prefixes': ['3.1', '3.2', '3.3', '3.4', '3.5', '3.6'],
    },
    'Labor Rights': {
        'file': 'role3_labor_criminal_justice.xlsx',
        'sheet': 'Labor Rights Data',
        'prefixes': ['4.1', '4.2', '4.3', '4.4', '4.5'],
    },
    'Criminal Justice': {
        'file': 'role3_labor_criminal_justice.xlsx',
        'sheet': 'Criminal Justice Data',
        # 5.6 Police Accountability is optional and currently empty; excluded.
        'prefixes': ['5.1', '5.2', '5.3', '5.4', '5.5'],
    },
}

PREDICTOR_FILE  = 'pm_predictor_variables (1).xlsx'
PREDICTOR_SHEET = 'Predictor Variables Data'

## 2. Load raw sub-indicator data

Headers live on row 3 (1-indexed) and state data on rows 4-54. We read with
`header=2` so pandas picks up the correct header row, then resolve each prefix
to the full column name.

In [3]:
def load_sheet(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=2)
    # First col is 'State'; drop any fully-empty rows
    df = df.dropna(subset=['State']).reset_index(drop=True)
    return df

def resolve_col(df, prefix):
    matches = [c for c in df.columns if str(c).startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Prefix {prefix!r} matched {len(matches)} columns: {matches}')
    return matches[0]

raw_frames = {}
for dim, cfg in DIMENSIONS.items():
    df = load_sheet(cfg['file'], cfg['sheet'])
    col_map = {resolve_col(df, p): f'{p}' for p in cfg['prefixes']}
    sub = df[['State'] + list(col_map.keys())].rename(columns=col_map)
    # Coerce any stray strings (e.g. 'NaN') to real NaN
    for c in col_map.values():
        sub[c] = pd.to_numeric(sub[c], errors='coerce')
    raw_frames[dim] = sub
    print(f'{dim:22s} {sub.shape[0]} states, {len(col_map)} sub-indicators')

Reproductive Rights    51 states, 6 sub-indicators
LGBTQ+ Protections     51 states, 5 sub-indicators
Voting Access          51 states, 6 sub-indicators
Labor Rights           51 states, 5 sub-indicators
Criminal Justice       51 states, 5 sub-indicators


## 3. Build the wide sub-indicator matrix

Merge all dimension frames on `State` to get one row per state and one column per
sub-indicator. Column names are prefixed with the dimension index (e.g. `1.1`, `2.1`)
so they remain unambiguous after merge.

In [4]:
wide = None
for dim, sub in raw_frames.items():
    wide = sub if wide is None else wide.merge(sub, on='State', how='outer')

print('States in wide matrix:', len(wide))
print('Sub-indicator columns:', [c for c in wide.columns if c != 'State'])
wide.head()

States in wide matrix: 51
Sub-indicator columns: ['1.1', '1.2', '1.3', '1.4', '1.5', '1.6', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1', '3.2', '3.3', '3.4', '3.5', '3.6', '4.1', '4.2', '4.3', '4.4', '4.5', '5.1', '5.2', '5.3', '5.4', '5.5']


,State,1.1,1.2,1.3,1.4,1.5,1.6,2.1,2.2,2.3,2.4,2.5,3.1,3.2,3.3,3.4,3.5,3.6,4.1,4.2,4.3,4.4,4.5,5.1,5.2,5.3,5.4,5.5
0,Alabama,0,50,0,0,0,0,-10.25,0,0,0,0,-2,0,0,0,33,0,9.41,7.25,0,0,0,0,0,0,0.0,50
1,Alaska,100,100,100,0,0,100,8.50,50,0,50,100,0,100,100,67,67,100,53.70,13.00,0,100,100,100,33,50,0.0,0
2,Arizona,80,100,100,100,100,0,5.75,50,0,50,0,-2,0,0,33,67,0,49.54,15.15,0,100,0,0,0,50,0.0,50
3,Arkansas,0,0,0,0,0,0,-10.50,0,0,0,0,-4,0,0,0,33,0,23.22,11.00,0,0,0,0,33,0,100.0,0
4,California,80,100,100,100,100,100,45.00,100,100,100,100,6,100,100,100,67,100,85.45,16.90,100,100,100,50,67,50,100.0,0


## 4. Min-max normalization to 0-100

For each sub-indicator we rescale to `[0, 100]` using
`(x - min) / (max - min) * 100`. Sub-indicators that are already discrete on a
0-100 scale pass through unchanged when their observed min=0 and max=100.

In [5]:
def minmax_100(s):
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.full(len(s), 50.0), index=s.index)
    return (s - lo) / (hi - lo) * 100.0

sub_cols = [c for c in wide.columns if c != 'State']
norm = wide[['State']].copy()
for c in sub_cols:
    norm[f'{c}_n'] = minmax_100(wide[c])

# Quick sanity: every normalized column should hit 0 and 100 (unless ties at ends)
summary = pd.DataFrame({
    'raw_min':  [wide[c].min() for c in sub_cols],
    'raw_max':  [wide[c].max() for c in sub_cols],
    'norm_min': [norm[f'{c}_n'].min() for c in sub_cols],
    'norm_max': [norm[f'{c}_n'].max() for c in sub_cols],
    'n_missing':[wide[c].isna().sum() for c in sub_cols],
}, index=sub_cols)
summary

,raw_min,raw_max,norm_min,norm_max,n_missing
1.1,0.00,100.00,0.0,100.0,0
1.2,0.00,100.00,0.0,100.0,0
1.3,0.00,100.00,0.0,100.0,0
1.4,0.00,100.00,0.0,100.0,0
1.5,0.00,100.00,0.0,100.0,0
1.6,0.00,100.00,0.0,100.0,0
2.1,-14.00,45.50,0.0,100.0,0
2.2,0.00,100.00,0.0,100.0,0
2.3,0.00,100.00,0.0,100.0,0
2.4,0.00,100.00,0.0,100.0,0


## 5. Dimension scores

Each dimension score = simple mean of its normalized sub-indicators (missing values
skipped with `mean(skipna=True)`, which is the `pandas` default).

In [6]:
dim_scores = norm[['State']].copy()
for dim, cfg in DIMENSIONS.items():
    cols = [f'{p}_n' for p in cfg['prefixes']]
    dim_scores[dim] = norm[cols].mean(axis=1)

dim_scores.head()

,State,Reproductive Rights,LGBTQ+ Protections,Voting Access,Labor Rights,Criminal Justice
0,Alabama,8.333333,1.260504,12.642857,4.156235,24.962406
1,Alaska,66.666667,47.563025,81.857143,63.965491,39.924812
2,Arizona,80.000000,26.638655,23.809524,46.308124,34.962406
3,Arkansas,0.000000,1.176471,10.261905,13.390900,29.924812
4,California,96.666667,99.831933,94.500000,97.819231,55.037594


## 6. Composite index — equal weights

Equal-weight composite: each of the 5 dimensions contributes 20%.

In [7]:
DIM_COLS = list(DIMENSIONS.keys())
dim_scores['Composite_EqualWeight'] = dim_scores[DIM_COLS].mean(axis=1)

ranked_eq = dim_scores.sort_values('Composite_EqualWeight', ascending=False).reset_index(drop=True)
ranked_eq['Rank_EqualWeight'] = ranked_eq.index + 1
ranked_eq[['Rank_EqualWeight', 'State', 'Composite_EqualWeight'] + DIM_COLS].head(10)

,Rank_EqualWeight,State,Composite_EqualWeight,Reproductive Rights,LGBTQ+ Protections,Voting Access,Labor Rights,Criminal Justice
0,1,District of Columbia,90.173669,83.333333,98.487395,94.047619,100.000000,75.000000
1,2,California,88.771085,96.666667,99.831933,94.500000,97.819231,55.037594
2,3,New Jersey,87.495115,100.000000,98.739496,70.690476,93.008010,75.037594
3,4,Washington,86.909839,96.666667,98.235294,88.547619,96.062023,55.037594
4,5,Colorado,86.108197,91.666667,100.000000,83.047619,90.789104,65.037594
5,6,Maryland,85.510690,83.333333,98.907563,90.928571,89.346387,65.037594
6,7,Minnesota,85.367700,100.000000,97.142857,89.738095,84.919952,55.037594
7,8,New York,84.933055,96.666667,99.663866,67.404762,95.892389,65.037594
8,9,Oregon,84.911211,100.000000,97.310924,87.357143,94.850395,45.037594
9,10,Illinois,84.188331,96.666667,88.823529,88.547619,71.866245,75.037594


## 7. Composite index — PCA weights

Run PCA on the 5 dimension scores (standardized) and use the absolute values of the
first principal component's loadings, normalized to sum to 1, as empirical weights.
This lets us sanity-check whether the data itself supports roughly equal weighting.

In [8]:
X = dim_scores[DIM_COLS].values
X_std = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
pca.fit(X_std)

loadings = pd.DataFrame(
    pca.components_.T,
    index=DIM_COLS,
    columns=[f'PC{i+1}' for i in range(5)],
)
explained = pd.Series(pca.explained_variance_ratio_, index=loadings.columns, name='explained_var')
print('Explained variance ratio:')
print(explained.round(3))
print()
print('Loadings:')
loadings.round(3)

Explained variance ratio:
PC1    0.832
PC2    0.087
PC3    0.044
PC4    0.019
PC5    0.017
Name: explained_var, dtype: float64

Loadings:


,PC1,PC2,PC3,PC4,PC5
Reproductive Rights,0.461,-0.340,-0.224,0.246,0.749
LGBTQ+ Protections,0.470,0.102,0.141,0.742,-0.445
Voting Access,0.445,-0.319,0.724,-0.415,-0.066
Labor Rights,0.455,-0.205,-0.637,-0.398,-0.432
Criminal Justice,0.402,0.855,0.011,-0.241,0.223


In [9]:
pc1 = loadings['PC1']
# If PC1 loads negatively on most dims, flip sign so "higher = more rights"
if (pc1 < 0).sum() > (pc1 > 0).sum():
    pc1 = -pc1

pca_weights = pc1.abs() / pc1.abs().sum()
print('PCA-derived weights (sum to 1):')
print(pca_weights.round(3))

dim_scores['Composite_PCA'] = dim_scores[DIM_COLS].mul(pca_weights, axis=1).sum(axis=1)

ranked_pca = dim_scores.sort_values('Composite_PCA', ascending=False).reset_index(drop=True)
ranked_pca['Rank_PCA'] = ranked_pca.index + 1
ranked_pca[['Rank_PCA', 'State', 'Composite_PCA']].head(10)

PCA-derived weights (sum to 1):
Reproductive Rights    0.206
LGBTQ+ Protections     0.211
Voting Access          0.199
Labor Rights           0.204
Criminal Justice       0.180
Name: PC1, dtype: float64


,Rank_PCA,State,Composite_PCA
0,1,District of Columbia,90.551323
1,2,California,89.637992
2,3,New Jersey,87.978572
3,4,Washington,87.758713
4,5,Colorado,86.730534
5,6,Minnesota,86.185207
6,7,Maryland,86.054987
7,8,Oregon,85.967259
8,9,New York,85.616761
9,10,Illinois,84.452062


## 8. Compare equal-weight vs PCA-weighted rankings

Spearman rank correlation; if it's very high (>0.98) the two methods agree and
equal weighting is defensible on empirical grounds.

In [10]:
compare = (
    dim_scores[['State']]
    .assign(
        Rank_EqualWeight = dim_scores['Composite_EqualWeight'].rank(ascending=False).astype(int),
        Rank_PCA         = dim_scores['Composite_PCA'].rank(ascending=False).astype(int),
        Composite_EqualWeight = dim_scores['Composite_EqualWeight'].round(2),
        Composite_PCA         = dim_scores['Composite_PCA'].round(2),
    )
    .assign(Rank_Diff = lambda d: d['Rank_EqualWeight'] - d['Rank_PCA'])
    .sort_values('Rank_EqualWeight')
    .reset_index(drop=True)
)

spearman = dim_scores[['Composite_EqualWeight', 'Composite_PCA']].corr(method='spearman').iloc[0, 1]
print(f'Spearman rank correlation (equal vs PCA): {spearman:.4f}')
print()
print('Biggest rank differences:')
print(compare.reindex(compare['Rank_Diff'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

Spearman rank correlation (equal vs PCA): 0.9993

Biggest rank differences:
         State  Rank_EqualWeight  Rank_PCA  Composite_EqualWeight  Composite_PCA  Rank_Diff
     Wisconsin                26        28                  39.34          38.77         -2
        Oregon                 9         8                  84.91          85.97          1
       Montana                27        26                  38.98          39.39          1
      Missouri                31        30                  33.26          33.37          1
South Carolina                41        42                  13.59          13.09         -1
          Iowa                30        31                  33.35          33.04         -1
     Louisiana                45        46                  13.02          12.48         -1
         Idaho                46        45                  12.95          12.84          1
      Kentucky                42        41                  13.56          13.55          1
    

## 9. Merge predictor variables

Trifecta (P10) is categorical — kept as a string; encode downstream as needed.

In [11]:
preds = pd.read_excel(PREDICTOR_FILE, sheet_name=PREDICTOR_SHEET, header=2)
preds = preds.dropna(subset=['State']).reset_index(drop=True)
# Rename predictor columns to short machine-friendly names
pred_rename = {}
for c in preds.columns:
    s = str(c)
    if s.startswith('P'):
        # 'P1 Trump Vote\nShare 2024 (%)' -> 'P1_TrumpShare2024'
        short = s.split(' ')[0]
        pred_rename[c] = short
preds = preds.rename(columns=pred_rename)
pred_cols = [c for c in preds.columns if c.startswith('P')]
print('Predictor columns:', pred_cols)
preds[['State'] + pred_cols].head()

Predictor columns: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10']


,State,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10
0,Alabama,64.8,59610,59.0,63.1,26.6,4.8,0.4847,3.0,1,R
1,Alaska,55.5,86370,66.0,58.8,3.2,7.5,0.4220,5.2,1,R
2,Arizona,52.8,72581,89.8,53.4,5.0,31.4,0.4680,3.6,1,R
3,Arkansas,65.5,56335,56.2,68.5,15.2,8.6,0.4780,3.6,1,R
4,California,37.7,91905,95.0,34.7,5.7,40.2,0.4890,5.3,1,D


## 10. Build master dataframe & export

Master = state + raw sub-indicators + normalized sub-indicators + dimension scores +
both composites + predictors. Saved to `data/processed/master_scored.csv`.

In [12]:
master = (
    wide
    .merge(norm, on='State', how='left')
    .merge(dim_scores, on='State', how='left')
    .merge(preds[['State'] + pred_cols], on='State', how='left')
)

master['Rank_EqualWeight'] = master['Composite_EqualWeight'].rank(ascending=False).astype(int)
master['Rank_PCA']         = master['Composite_PCA'].rank(ascending=False).astype(int)

out_path = OUT_DIR / 'master_scored.csv'
master.to_csv(out_path, index=False)
print(f'Wrote {out_path}  ({master.shape[0]} rows x {master.shape[1]} cols)')
master[['State', 'Rank_EqualWeight', 'Composite_EqualWeight'] + DIM_COLS].sort_values('Rank_EqualWeight').head(15)

Wrote /Users/latahviawilliams/DSS_rights/data/processed/master_scored.csv  (51 rows x 74 cols)


,State,Rank_EqualWeight,Composite_EqualWeight,Reproductive Rights,LGBTQ+ Protections,Voting Access,Labor Rights,Criminal Justice
8,District of Columbia,1,90.173669,83.333333,98.487395,94.047619,100.000000,75.000000
4,California,2,88.771085,96.666667,99.831933,94.500000,97.819231,55.037594
30,New Jersey,3,87.495115,100.000000,98.739496,70.690476,93.008010,75.037594
47,Washington,4,86.909839,96.666667,98.235294,88.547619,96.062023,55.037594
5,Colorado,5,86.108197,91.666667,100.000000,83.047619,90.789104,65.037594
20,Maryland,6,85.510690,83.333333,98.907563,90.928571,89.346387,65.037594
23,Minnesota,7,85.367700,100.000000,97.142857,89.738095,84.919952,55.037594
32,New York,8,84.933055,96.666667,99.663866,67.404762,95.892389,65.037594
37,Oregon,9,84.911211,100.000000,97.310924,87.357143,94.850395,45.037594
13,Illinois,10,84.188331,96.666667,88.823529,88.547619,71.866245,75.037594


## 11. Top 10 / Bottom 10 snapshot

In [13]:
snapshot = master[['State', 'Rank_EqualWeight', 'Composite_EqualWeight'] + DIM_COLS].copy()
snapshot[DIM_COLS] = snapshot[DIM_COLS].round(1)
snapshot['Composite_EqualWeight'] = snapshot['Composite_EqualWeight'].round(2)
snapshot = snapshot.sort_values('Rank_EqualWeight')

print('TOP 10')
print(snapshot.head(10).to_string(index=False))
print()
print('BOTTOM 10')
print(snapshot.tail(10).to_string(index=False))

TOP 10
               State  Rank_EqualWeight  Composite_EqualWeight  Reproductive Rights  LGBTQ+ Protections  Voting Access  Labor Rights  Criminal Justice
District of Columbia                 1                  90.17                 83.3                98.5           94.0         100.0              75.0
          California                 2                  88.77                 96.7                99.8           94.5          97.8              55.0
          New Jersey                 3                  87.50                100.0                98.7           70.7          93.0              75.0
          Washington                 4                  86.91                 96.7                98.2           88.5          96.1              55.0
            Colorado                 5                  86.11                 91.7               100.0           83.0          90.8              65.0
            Maryland                 6                  85.51                 83.3           